In [2]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.two_tower_training import (  # noqa: E402
    TrainConfig,
    compare_on_common_objectives,
    load_processed_two_tower_data,
    summarize_results,
    train_one_config,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PREFIX = "vehicle_sensor_subset_200ms_expanded_features"
OUT_DIR = PROJECT_ROOT / "experiments" / "static_action_two_tower"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DEVICE: {DEVICE}")

# Load the full expanded dataset, then remove all live/time-varying columns from A_examples.
data = load_processed_two_tower_data(PROCESSED_DIR, prefix=PREFIX, utility_name="saved")
meta = data["meta"]
action_names = list(meta["action_feature_names"])
examples_index = data["examples_index"]

mask_names = [name for name in action_names if name.startswith("mask_n")]
canonical_slot_names = []
for slot in (1, 2, 3):
    canonical_slot_names.extend([
        f"slot{slot}_sensor_x_norm",
        f"slot{slot}_sensor_y_norm",
    ])
geometry_names = [
    "pairwise_dist_1_m",
    "pairwise_dist_2_m",
    "pairwise_dist_3_m",
    "subset_centroid_x_norm",
    "subset_centroid_y_norm",
    "subset_max_spread_m",
    "subset_triangle_area_norm",
    "subset_size",
]
static_action_names = mask_names + canonical_slot_names + geometry_names

missing = sorted(set(static_action_names) - set(action_names))
if missing:
    raise ValueError(f"Missing expected static action columns: {missing}")

subset_key = "subset_str" if "subset_str" in examples_index.columns else "subset"
subset_labels = examples_index[subset_key].astype(str).to_numpy()
n_unique_subsets = int(pd.Series(subset_labels).nunique())
expected_actions = int(meta.get("num_actions_per_time", len(meta.get("full_subset_universe", []))))

context_names = list(meta["context_feature_names"])
sensor_xy = {}
for node in meta["ordered_nodes"]:
    x_name = f"n{int(node)}_sensor_x_norm"
    y_name = f"n{int(node)}_sensor_y_norm"
    sensor_xy[int(node)] = (
        float(data["C_by_time"][0, context_names.index(x_name)]),
        float(data["C_by_time"][0, context_names.index(y_name)]),
    )

def parse_subset_nodes(label: str) -> list[int]:
    cleaned = str(label).replace("(", "").replace(")", "").replace(",", "-").replace(" ", "")
    return [int(part) for part in cleaned.split("-") if part]

# Rebuild slot coordinates in canonical subset order. The raw slot*_sensor_* columns
# come from the live selected-node descriptor ordering, so they drift across time.
unique_subset_labels = pd.Series(subset_labels).drop_duplicates().astype(str).tolist()
slot_lookup = {}
for label in unique_subset_labels:
    nodes = parse_subset_nodes(label)
    vals = []
    for slot_idx in range(3):
        if slot_idx < len(nodes):
            vals.extend(sensor_xy[nodes[slot_idx]])
        else:
            vals.extend([0.0, 0.0])
    slot_lookup[label] = vals
canonical_slot_matrix = np.asarray([slot_lookup[label] for label in subset_labels], dtype=np.float32)

A_static_parts = []
for name in mask_names:
    A_static_parts.append(data["A_examples"][:, action_names.index(name)].astype(np.float32))
for j, _name in enumerate(canonical_slot_names):
    A_static_parts.append(canonical_slot_matrix[:, j])
for name in geometry_names:
    A_static_parts.append(data["A_examples"][:, action_names.index(name)].astype(np.float32))
A_static = np.column_stack(A_static_parts).astype(np.float32)

# The same subset must have exactly the same action vector at every timestamp.
static_df = pd.DataFrame(A_static, columns=static_action_names)
static_df["subset_key"] = subset_labels
grouped_max = static_df.groupby("subset_key", sort=False)[static_action_names].max()
grouped_min = static_df.groupby("subset_key", sort=False)[static_action_names].min()
max_static_drift = float((grouped_max - grouped_min).to_numpy().max())
if max_static_drift > 1e-6:
    raise AssertionError(f"Static action features drift across time: max drift={max_static_drift:.3g}")

first_time_id = int(examples_index["time_id"].min())
catalog_mask = examples_index["time_id"].eq(first_time_id).to_numpy()
A_static_catalog = A_static[catalog_mask].astype(np.float32)
catalog_cols = [col for col in ["time_id", "subset", "subset_str", "subset_size"] if col in examples_index.columns]
action_catalog_df = examples_index.loc[catalog_mask, catalog_cols].reset_index(drop=True).copy()
for j, name in enumerate(static_action_names):
    action_catalog_df[name] = A_static_catalog[:, j]

if len(action_catalog_df) != n_unique_subsets:
    raise AssertionError(f"Expected one catalog row per subset, got {len(action_catalog_df)} vs {n_unique_subsets}")
if expected_actions and n_unique_subsets != expected_actions:
    raise AssertionError(f"Expected {expected_actions} actions per time, got {n_unique_subsets}")

static_meta = dict(meta)
static_meta["action_feature_names"] = static_action_names
static_meta["action_raw_dim"] = int(A_static.shape[1])
static_meta["static_action_feature_names"] = static_action_names
static_meta["static_action_source_prefix"] = PREFIX
static_meta["static_action_note"] = (
    "Action tower sees only subset membership, selected sensor coordinates, subset geometry, "
    "and subset size. All RSSI/audio/rank/acoustic-COM features remain in the context side "
    "or are removed from the action tower so action embeddings can be precomputed."
)

static_data = dict(data)
static_data["A_examples"] = A_static
static_data["meta"] = static_meta

print("\nStatic action feature check")
print(f"original action_dim: {len(action_names)}")
print(f"static action_dim:   {A_static.shape[1]}")
print(f"unique subsets:      {n_unique_subsets}")
print(f"max static drift:    {max_static_drift:.3g}")
print("static action columns:")
print(static_action_names)

config = TrainConfig(
    run_name="static_action_saved_h512_d2_e16_lr5e4",
    utility_name="saved",
    hidden=512,
    emb_dim=16,
    depth=2,
    dropout=0.05,
    combine_mode="mul_only",
    loss_name="mse",
    lr=5e-4,
    weight_decay=1e-4,
    batch_size=8192,
    max_epochs=300,
    patience=300,
    seed=22,
    log_every=1,
    num_workers=0,
)

result = train_one_config(static_data, config, device=DEVICE, verbose=True)
summary_df = summarize_results([result])
common_eval_df = compare_on_common_objectives([result])
test_eval_df = (
    common_eval_df[common_eval_df["split"].eq("test")]
    .sort_values(["eval_objective", "avg_norm_regret", "mean_rank"])
    .reset_index(drop=True)
)

display(summary_df)
display(test_eval_df)

# Precompute exactly one embedding per candidate subset. At runtime only the context tower is live.
prepared = result["prepared"]
A_mu = prepared["standardizers"]["A_mu"]
A_sigma = prepared["standardizers"]["A_sigma"]
A_static_catalog_std = ((A_static_catalog - A_mu) / A_sigma).astype(np.float32)

model = result["model"]
model.eval()
with torch.no_grad():
    A_tensor = torch.from_numpy(A_static_catalog_std).to(result["device"])
    static_action_embeddings = model.embed_action(A_tensor).cpu().numpy().astype(np.float32)

run_dir = OUT_DIR / config.run_name
run_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = run_dir / "checkpoint.pt"
embedding_path = run_dir / "static_action_embeddings.npz"
history_path = run_dir / "history.csv"
metrics_path = run_dir / "metrics.json"
summary_path = run_dir / "summary.csv"
eval_path = run_dir / "common_eval.csv"
catalog_path = run_dir / "static_action_catalog.csv"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": asdict(config),
        "context_dim": int(prepared["C_by_time"].shape[1]),
        "action_dim": int(prepared["A_examples"].shape[1]),
        "static_action_feature_names": static_action_names,
        "standardizers": prepared["standardizers"],
        "meta": static_meta,
        "best_epoch": int(result["best_epoch"]),
    },
    checkpoint_path,
)
np.savez_compressed(
    embedding_path,
    action_embeddings=static_action_embeddings,
    action_vectors=A_static_catalog,
    action_vectors_std=A_static_catalog_std,
    subset_key=action_catalog_df[subset_key].astype(str).to_numpy() if subset_key in action_catalog_df else action_catalog_df["subset"].astype(str).to_numpy(),
    static_action_feature_names=np.array(static_action_names),
)
result["history"].to_csv(history_path, index=False)
summary_df.to_csv(summary_path, index=False)
common_eval_df.to_csv(eval_path, index=False)
action_catalog_df.to_csv(catalog_path, index=False)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(result["metrics"], f, indent=2)

print("\nSaved static-action run")
print(f"run_dir: {run_dir}")
print(f"checkpoint: {checkpoint_path}")
print(f"static action embeddings: {embedding_path}")
print(f"embedding table shape: {static_action_embeddings.shape} = {n_unique_subsets} subsets x {config.emb_dim} dims")


PROJECT_ROOT: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt
DEVICE: cuda

Static action feature check
original action_dim: 214
static action_dim:   20
unique subsets:      41
max static drift:    0
static action columns:
['mask_n11', 'mask_n12', 'mask_n13', 'mask_n14', 'mask_n15', 'mask_n16', 'slot1_sensor_x_norm', 'slot1_sensor_y_norm', 'slot2_sensor_x_norm', 'slot2_sensor_y_norm', 'slot3_sensor_x_norm', 'slot3_sensor_y_norm', 'pairwise_dist_1_m', 'pairwise_dist_2_m', 'pairwise_dist_3_m', 'subset_centroid_x_norm', 'subset_centroid_y_norm', 'subset_max_spread_m', 'subset_triangle_area_norm', 'subset_size']
static_action_saved_h512_d2_e16_lr5e4 ep 001/300* loss=0.07355 val_rmse=0.1800 top1=0.131 top3=0.361 reg=0.0787 rank=9.28 13s
static_action_saved_h512_d2_e16_lr5e4 ep 002/300* loss=0.02432 val_rmse=0.1610 top1=0.309 top3=0.413 reg=0.0583 rank=7.72 24s
static_action_saved_h512_d2_e16_lr5e4 ep 003/300* loss=0.01670 val_rmse=0.1466 top1=0.311 top3=0.475 reg=0.0357 ran

,run_name,utility,hidden,emb_dim,depth,dropout,combine,loss,best_epoch,train_rmse,...,val_avg_regret,val_avg_norm_regret,test_rmse,test_mae,test_r2,test_top1,test_top3,test_mean_rank,test_avg_regret,test_avg_norm_regret
0,static_action_saved_h512_d2_e16_lr5e4,saved,512,16,2,0.05,mul_only,mse,190,0.006733,...,0.015747,0.032633,0.116167,0.081179,0.65928,0.444072,0.657736,3.722706,0.021077,0.040005


,run_name,train_utility,eval_objective,split,hidden,emb_dim,combine,top1,top3,mean_rank,avg_regret,avg_norm_regret
0,static_action_saved_h512_d2_e16_lr5e4,saved,contains_closest,test,512,16,mul_only,0.949096,0.949096,1.814468,0.050904,0.050904
1,static_action_saved_h512_d2_e16_lr5e4,saved,rank_discount,test,512,16,mul_only,0.949096,0.949096,1.961822,0.027685,0.033222
2,static_action_saved_h512_d2_e16_lr5e4,saved,saved_rational,test,512,16,mul_only,0.444072,0.657736,3.722706,0.021077,0.040005



Saved static-action run
run_dir: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\experiments\static_action_two_tower\static_action_saved_h512_d2_e16_lr5e4
checkpoint: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\experiments\static_action_two_tower\static_action_saved_h512_d2_e16_lr5e4\checkpoint.pt
static action embeddings: d:\2023-2028_UCLA_Research_Projects\IoBT\17-5-26- MILCOM Attempt\experiments\static_action_two_tower\static_action_saved_h512_d2_e16_lr5e4\static_action_embeddings.npz
embedding table shape: (41, 16) = 41 subsets x 16 dims


In [ ]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import copy
import json
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from scripts.two_tower_training import (  # noqa: E402
    TwoTowerMLP,
    TrainConfig,
    build_utility_labels,
    compare_on_common_objectives,
    evaluate_split,
    make_loader,
    make_loss,
    prepare_standardized_data,
    summarize_results,
)

# Future-reference training cell.
# It keeps the first cell's model/training config the same, but changes best-epoch selection.
# The original helper selected by (val_avg_regret, val_rmse). That is defensible, but it
# can prefer an epoch with slightly lower regret while later epochs have better rank/top-k.
#
# New selector:
#   1. Keep any epoch within REGRET_REL_TOL of the best validation regret so far.
#   2. Within that regret-tolerant set, prefer lower mean_rank.
#   3. Then prefer higher top3, higher top1, lower normalized regret, lower RMSE.
#
# This keeps saved utility regret primary, but avoids overfitting the checkpoint choice to
# tiny regret differences when subset decision metrics are clearly better.

required_globals = [
    "static_data",
    "static_action_names",
    "A_static_catalog",
    "action_catalog_df",
    "subset_key",
    "OUT_DIR",
    "DEVICE",
    "config",
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError(
        "Run the first cell first so static_data/config/catalog variables exist. "
        f"Missing: {missing_globals}"
    )

REGRET_REL_TOL = 0.03  # 3% regret tolerance; keep regret primary, then improve decision quality.
RUN_NAME = f"{config.run_name}_regret_tol_rank_selected"
RUN_DIR = OUT_DIR / RUN_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Same config as first cell, except run_name so artifacts do not overwrite the original run.
select_cfg = copy.deepcopy(config)
select_cfg.run_name = RUN_NAME

print("Future-reference static-action run")
print("same hyperparameters as first cell")
print("old selector: (val_avg_regret, val_rmse)")
print(f"new selector: regret within {REGRET_REL_TOL:.1%} of best-so-far -> mean_rank -> top3 -> top1 -> norm_regret -> rmse")
print("run_dir:", RUN_DIR)


def selector_key(val: dict[str, float], best_regret_so_far: float) -> tuple[float, ...]:
    """Lower is better. Regret remains primary, but only outside a tolerance band."""
    regret = float(val["avg_regret"])
    regret_floor = best_regret_so_far * (1.0 + REGRET_REL_TOL)
    regret_penalty = max(0.0, regret - regret_floor)
    return (
        regret_penalty,
        float(val["mean_rank"]),
        -float(val["top3"]),
        -float(val["top1"]),
        float(val["avg_norm_regret"]),
        float(val["rmse"]),
    )


def save_selected_checkpoint(
    *,
    path: Path,
    epoch: int,
    key: tuple[float, ...],
    val: dict[str, float],
    model: TwoTowerMLP,
    optimizer: torch.optim.Optimizer,
    prepared: dict,
    config: TrainConfig,
    best_regret_so_far: float,
) -> None:
    torch.save(
        {
            "epoch": int(epoch),
            "selector_key": [float(x) for x in key],
            "best_regret_so_far": float(best_regret_so_far),
            "regret_rel_tol": float(REGRET_REL_TOL),
            "val_metrics": {k: float(v) for k, v in val.items()},
            "model_state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
            "optimizer_state_dict": optimizer.state_dict(),
            "config": asdict(config),
            "context_dim": int(prepared["C_by_time"].shape[1]),
            "action_dim": int(prepared["A_examples"].shape[1]),
            "static_action_feature_names": list(static_action_names),
            "standardizers": prepared["standardizers"],
            "meta": prepared["meta"],
            "selection_rule": (
                f"avg_regret within {REGRET_REL_TOL:.1%} of best-so-far, then "
                "mean_rank, top3, top1, avg_norm_regret, rmse"
            ),
        },
        path,
    )


# Prepare data exactly as train_one_config would.
train_data = dict(static_data)
if train_data.get("utility_name") != select_cfg.utility_name or train_data.get("utility_kwargs") != (select_cfg.utility_kwargs or {}):
    train_data["y_examples"] = build_utility_labels(
        train_data["examples_index"],
        train_data["saved_y_examples"],
        train_data["meta"],
        select_cfg.utility_name,
        select_cfg.utility_kwargs or {},
    ).astype(np.float32)
    train_data["utility_name"] = select_cfg.utility_name
    train_data["utility_kwargs"] = select_cfg.utility_kwargs or {}

prepared_select = prepare_standardized_data(train_data, select_cfg)
train_loader = make_loader(prepared_select, prepared_select["split"]["train"], select_cfg, shuffle=True)

model_select = TwoTowerMLP(
    context_dim=prepared_select["C_by_time"].shape[1],
    action_dim=prepared_select["A_examples"].shape[1],
    hidden=select_cfg.hidden,
    emb_dim=select_cfg.emb_dim,
    depth=select_cfg.depth,
    dropout=select_cfg.dropout,
    combine_mode=select_cfg.combine_mode,
).to(DEVICE)

optimizer = torch.optim.AdamW(model_select.parameters(), lr=select_cfg.lr, weight_decay=select_cfg.weight_decay)
loss_fn = make_loss(select_cfg.loss_name)

best_state = None
best_epoch = -1
best_key = None
best_regret_so_far = float("inf")
wait = 0
history_rows = []
start_time = time.time()

for epoch in range(1, select_cfg.max_epochs + 1):
    model_select.train()
    losses = []

    for C, A, y in train_loader:
        C = C.to(DEVICE)
        A = A.to(DEVICE)
        y = y.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        pred = model_select(C, A)
        loss = loss_fn(pred, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_select.parameters(), max_norm=5.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))

    train_loss = float(np.mean(losses))
    val = evaluate_split(model_select, prepared_select, "val", select_cfg, DEVICE)

    best_regret_so_far = min(best_regret_so_far, float(val["avg_regret"]))
    key = selector_key(val, best_regret_so_far)
    improved = best_key is None or key < best_key

    if improved:
        best_key = key
        best_epoch = int(epoch)
        best_state = {k: v.detach().cpu().clone() for k, v in model_select.state_dict().items()}
        wait = 0
        save_selected_checkpoint(
            path=CHECKPOINT_DIR / "best_by_regret_tolerant_rank.pt",
            epoch=epoch,
            key=key,
            val=val,
            model=model_select,
            optimizer=optimizer,
            prepared=prepared_select,
            config=select_cfg,
            best_regret_so_far=best_regret_so_far,
        )
    else:
        wait += 1

    history_rows.append(
        {
            "epoch": int(epoch),
            "train_loss": train_loss,
            "best_regret_so_far": float(best_regret_so_far),
            "selector_regret_penalty": float(key[0]),
            "selector_mean_rank": float(key[1]),
            "selector_neg_top3": float(key[2]),
            "selector_neg_top1": float(key[3]),
            "is_best_epoch_so_far": bool(improved),
            **{f"val_{k}": float(v) for k, v in val.items()},
        }
    )

    should_log = epoch == 1 or epoch % select_cfg.log_every == 0 or improved or epoch == select_cfg.max_epochs
    if should_log:
        star = "*" if improved else " "
        print(
            f"{select_cfg.run_name:>58s} ep {epoch:03d}/{select_cfg.max_epochs:03d}{star} "
            f"loss={train_loss:.5f} val_reg={val['avg_regret']:.4f} "
            f"rank={val['mean_rank']:.2f} top1={val['top1']:.3f} top3={val['top3']:.3f} "
            f"normreg={val['avg_norm_regret']:.4f} rmse={val['rmse']:.4f} "
            f"{time.time() - start_time:.0f}s"
        )

    if wait >= select_cfg.patience:
        print(f"early stop at epoch {epoch}; best epoch {best_epoch}")
        break

if best_state is None:
    raise RuntimeError("Training ended without a selected state.")

model_select.load_state_dict(best_state)

selected_result = {
    "config": select_cfg,
    "model": model_select,
    "prepared": prepared_select,
    "history": pd.DataFrame(history_rows),
    "metrics": {
        split: evaluate_split(model_select, prepared_select, split, select_cfg, DEVICE)
        for split in ("train", "val", "test")
    },
    "best_epoch": best_epoch,
    "device": DEVICE,
}

selected_summary_df = summarize_results([selected_result])
selected_common_eval_df = compare_on_common_objectives(
    [selected_result],
    objectives=[
        {"name": "saved_rational", "utility_name": "saved", "utility_kwargs": {}},
        {"name": "contains_closest", "utility_name": "closest_binary", "utility_kwargs": {}},
        {"name": "rank_discount", "utility_name": "rank_discount", "utility_kwargs": {}},
    ],
    split_names=("train", "val", "test"),
)

# Save the final selected checkpoint plus frozen static action cache.
A_mu = prepared_select["standardizers"]["A_mu"]
A_sigma = prepared_select["standardizers"]["A_sigma"]
A_static_catalog_std = ((A_static_catalog - A_mu) / A_sigma).astype(np.float32)

model_select.eval()
with torch.no_grad():
    A_tensor = torch.from_numpy(A_static_catalog_std).to(DEVICE)
    selected_static_action_embeddings = model_select.embed_action(A_tensor).cpu().numpy().astype(np.float32)

checkpoint_path = RUN_DIR / "selected_checkpoint.pt"
embedding_path = RUN_DIR / "static_action_embeddings.npz"
history_path = RUN_DIR / "history.csv"
summary_path = RUN_DIR / "summary.csv"
common_eval_path = RUN_DIR / "common_eval.csv"
manifest_path = RUN_DIR / "selection_manifest.json"

selected_payload = {
    "model_state_dict": model_select.state_dict(),
    "config": asdict(select_cfg),
    "context_dim": int(prepared_select["C_by_time"].shape[1]),
    "action_dim": int(prepared_select["A_examples"].shape[1]),
    "static_action_feature_names": list(static_action_names),
    "standardizers": prepared_select["standardizers"],
    "meta": prepared_select["meta"],
    "best_epoch": int(best_epoch),
    "best_key": [float(x) for x in best_key],
    "regret_rel_tol": float(REGRET_REL_TOL),
    "metrics": selected_result["metrics"],
}
torch.save(selected_payload, checkpoint_path)
np.savez_compressed(
    embedding_path,
    action_embeddings=selected_static_action_embeddings,
    action_vectors=A_static_catalog.astype(np.float32),
    action_vectors_std=A_static_catalog_std,
    subset_key=action_catalog_df[subset_key].astype(str).to_numpy(),
    static_action_feature_names=np.array(static_action_names, dtype=object),
)
selected_result["history"].to_csv(history_path, index=False)
selected_summary_df.to_csv(summary_path, index=False)
selected_common_eval_df.to_csv(common_eval_path, index=False)

manifest = {
    "run_name": select_cfg.run_name,
    "best_epoch": int(best_epoch),
    "same_hyperparameters_as_first_cell": True,
    "changed_only_epoch_selection": True,
    "regret_rel_tol": float(REGRET_REL_TOL),
    "selection_rule": (
        f"avg_regret within {REGRET_REL_TOL:.1%} of best-so-far, then "
        "mean_rank, top3, top1, avg_norm_regret, rmse"
    ),
    "checkpoint_path": str(checkpoint_path),
    "embedding_path": str(embedding_path),
    "history_path": str(history_path),
    "summary_path": str(summary_path),
    "common_eval_path": str(common_eval_path),
}
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSelected best epoch:", best_epoch)
print("Saved checkpoint:", checkpoint_path)
print("Saved static action embeddings:", embedding_path)
print("Embedding shape:", selected_static_action_embeddings.shape)

print("\nSummary:")
display(selected_summary_df)

print("\nTest metrics:")
display(
    selected_common_eval_df[selected_common_eval_df["split"].eq("test")][
        ["run_name", "eval_objective", "top1", "top3", "mean_rank", "avg_regret", "avg_norm_regret"]
    ].reset_index(drop=True)
)
